# Preparations

In [4]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    # Change this path if your code folder is elsewhere in Drive.
    PROJECT_DIR = Path("/content/drive/MyDrive/student/cafe_project")
    DATA_DIR = Path("/content/dataset")       # Local disk on the Colab server.
    SAVE_PATH = PROJECT_DIR / "runs"          # Keep results and weights in Drive.
else:
    PROJECT_DIR = Path('.').expanduser().resolve()
    DATA_DIR = Path("./dataset").expanduser().resolve()
    SAVE_PATH = Path("./runs").expanduser().resolve()

DATASET_ZIP = PROJECT_DIR / "dataset.zip"  # Set None if already extracted manually.
if not (PROJECT_DIR / "requirements.txt").is_file():
    raise FileNotFoundError("Set PROJECT_DIR to the code folder containing requirements.txt.")
print("Code:", PROJECT_DIR)
print("Dataset:", DATA_DIR)
print("Results:", SAVE_PATH)


Code: C:\Users\lucas\Downloads\cafe_project-20261118T022300Z-1-001\cafe_project
Dataset: C:\Users\lucas\Downloads\cafe_project-20261118T022300Z-1-001\cafe_project\dataset
Results: C:\Users\lucas\Downloads\cafe_project-20261118T022300Z-1-001\cafe_project\runs


## Install Packages
Use requirements.txt from the code folder configured above.

Added support Intel XPU scikit device, to use it, set `INSTALL_XPU=True` below before running this cell. It installs the Intel GPU build of PyTorch into the selected notebook kernel. After installation, set the option back to False, restart the kernel, and rerun the notebook. The configuration cell reports the PyTorch build and whether it detects XPU.


In [5]:
INSTALL_XPU = False  # Enable once to install the Intel GPU build of PyTorch.
if INSTALL_XPU:
    %pip install --force-reinstall torch --index-url https://download.pytorch.org/whl/xpu

requirements_path = str(PROJECT_DIR / "requirements.txt")
%pip install -r "{requirements_path}"


Note: you may need to restart the kernel to use updated packages.


## 1. Prepare the Data
Extract DATASET_ZIP into DATA_DIR and save results under SAVE_PATH, as configured in section 0.
For manually extracted data, set DATASET_ZIP=None and select its DATA_DIR in section 0.
Completed extraction is reused. Prepare the data again after starting a new Colab runtime.


In [6]:
from datetime import datetime
import os
import sys

SAVE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(SAVE_PATH / "matplotlib_cache"))


'C:\\Users\\lucas\\Downloads\\cafe_project-20261118T022300Z-1-001\\cafe_project\\runs\\matplotlib_cache'

In [7]:
from common.data_setup import prepare_dataset

# Reuse the dataset if this ZIP has already been extracted.
DATA_DIR = prepare_dataset(DATASET_ZIP, DATA_DIR)


c:\Users\lucas\anaconda3\envs\cafe_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset already extracted: C:\Users\lucas\Downloads\cafe_project-20261118T022300Z-1-001\cafe_project\dataset


In [8]:
import math
from pathlib import Path
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F

VIEWS = ("sw", "se", "ne", "nw")
COLORS = ("red", "blue", "green", "yellow", "black", "white")
PATTERNS = ("plain", "flower", "bird", "leaf", "stripes")


In [9]:
MODEL_NAME = 'improved'  # Set "improved" in your copied notebook.
MAX_LEVEL = 3          # Cumulative VAL scope: 1, 2, or 3. All TRAIN data remain available.
CHECKPOINT_PATH = (PROJECT_DIR / "weights/baseline.pth").resolve()
INPUT_VIEWS = ("sw",) if MODEL_NAME == "baseline" else VIEWS
xpu_available = hasattr(torch, "xpu") and torch.xpu.is_available()
DEVICE = "cuda" if torch.cuda.is_available() else ("xpu" if xpu_available else "cpu")
SEED = 17
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 0.0006
USE_WANDB = False      # Optional. False requires no W&B login.
WANDB_PROJECT = "cafe-cv-project"  # Used only when USE_WANDB=True.
WANDB_ENTITY = None
RUN_NAME = MODEL_NAME + "_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR = SAVE_PATH / RUN_NAME

print("Device:", DEVICE, "| Cameras:", INPUT_VIEWS)


Device: xpu | Cameras: ('sw', 'se', 'ne', 'nw')


## 2. Read the Data
L1: basic scenes where the target cup can be identified from SW.
L2: scenes containing cups with similar colors and patterns.
L3: scenes with occlusion and differences in what is visible across views.
TRAIN/VAL request lists are fixed for each level. Do not create new splits.
Training uses all 1,050 L1-L3 TRAIN requests. MAX_LEVEL selects only the cumulative VAL scope: 1 -> L1; 2 -> L1+L2; 3 -> L1+L2+L3.
VAL: L1 280 / L2 216 / L3 50 requests. TEST is private.


In [10]:
import json
import random
import time
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)


In [11]:
from common.request import parse_request


def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)


def scene_folder(row, data_root):
    return Path(data_root) / row["scene_path"]


def read_requests(data_root, max_level, split):
    """Use all supplied TRAIN requests and cumulative VAL levels."""
    if split not in ("train", "val"):
        raise ValueError("Use TRAIN/VAL only for training and model selection.")
    if max_level not in (1, 2, 3):
        raise ValueError("Choose a challenge level from 1 to 3.")
    index = read_json(Path(data_root) / "index.json")
    if index["schema"] != "cafe_three_level_v10":
        raise ValueError("Select the current three-level dataset folder.")
    if index.get("revision") != "l1_l2_l3_only_v1":
        raise ValueError("Use the updated L1-L3 dataset and an empty extraction folder.")
    rows = []
    last_level = 3 if split == "train" else max_level
    for level in range(1, last_level + 1):
        metadata = read_json(Path(data_root) / f"L{level}.json")
        rows.extend(metadata["splits"][split])
    if len({row["sample_id"] for row in rows}) != len(rows):
        raise ValueError("Duplicate request IDs in the dataset.")
    for row in rows:
        # Normalize legacy dataset wording; cup colors, patterns, and targets stay the same.
        row["request"] = {**row["request"], **parse_request(row["request"]["text"])}
    return sorted(rows, key=lambda row: row["sample_id"])


In [12]:
def load_observations(folder, views=VIEWS):
    """Read RGB, depth, and camera calibration for the specified views.

    Return observations[view]["rgb"] and observations[view]["geometry"].
    rgb: uint8 [H, W, 3], depth_m: float32 [H, W], depth_valid: bool [H, W].
    K: float64 [3, 3], camera_to_world: float64 [4, 4].
    Do not include ground truth, scene IDs, or levels in the returned observations.
    """
    scene_path = Path(folder)
    cameras = read_json(scene_path / "camera.json")
    observations = {}

    # Iterate over views because the number of requested views can vary.
    for view in views:
        rgb_path = scene_path / "rgb" / f"{view}.png"
        depth_path = scene_path / "depth" / f"{view}.tiff"
        valid_path = scene_path / "depth" / f"{view}_valid.png"

        with Image.open(rgb_path) as image:
            rgb_image = image.convert("RGB")
            rgb = np.array(rgb_image)

        # Read metric depth from TIFF; do not use depth_preview.png.
        with Image.open(depth_path) as image:
            depth_m = np.array(image, dtype=np.float32)

        with Image.open(valid_path) as image:
            valid_pixels = np.array(image)
        depth_valid = valid_pixels > 0

        camera = cameras[view]
        intrinsic_matrix = np.array(camera["K"], dtype=np.float64)
        camera_transform = np.array(camera["camera_to_world"], dtype=np.float64)

        geometry = {
            "depth_m": depth_m,
            "depth_valid": depth_valid,
            "K": intrinsic_matrix,
            "camera_to_world": camera_transform,
        }
        observations[view] = {"rgb": rgb, "geometry": geometry}

    return observations


def load_supervision(folder, request_id, view="sw"):
    """Read supervision for the requested cup. Never call this during inference.

    Return a copy of the selected view's entry in points.json.
    Load target_mask and target_parts as uint8 arrays instead of filenames.
    mask: target cup=255, other=0. parts: background=0, body=1, handle=2.
    When coordinate_loss_valid=False, reference_uv is None.
    """
    label_folder = Path(folder) / "label" / request_id
    labels_by_view = read_json(label_folder / "points.json")
    supervision = labels_by_view[view].copy()

    target_mask_path = label_folder / supervision["target_mask"]
    with Image.open(target_mask_path) as image:
        target_mask = np.array(image)

    target_parts_path = label_folder / supervision["target_parts"]
    with Image.open(target_parts_path) as image:
        target_parts = np.array(image)

    supervision["target_mask"] = target_mask
    supervision["target_parts"] = target_parts
    return supervision


In [13]:
def image_to_tensor(rgb):
    """uint8 [H, W, 3] -> float32 [3, H, W], preserving the original RGB array."""
    image_tensor = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float()
    normalized_image = (image_tensor / 255.0 - 0.5) / 0.25
    return normalized_image


In [14]:
class CupDataset(Dataset):
    def __init__(self, data_root, max_level=1, split="train", views=("sw",)):
        if split not in ("train", "val"):
            raise ValueError("Do not use TEST for training or checkpoint selection.")

        self.data_root = Path(data_root)
        # Use the request loader above without creating new splits.
        # Use all supplied TRAIN requests and the selected cumulative VAL levels.
        requests = read_requests(self.data_root, max_level, split)
        self.rows = [{**row, "view": view} for row in requests for view in views]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        """Build samples from one selected view, request IDs, and reference point labels.

        Return keys: image(float32 [3, 512, 512]), color(int), pattern(int),
        uv(float32 [2]), valid(bool). Coordinates are horizontal u, vertical v.
        Normalize valid points as (reference_uv + 0.5) / [W, H].
        Samples without point supervision use [NaN, NaN] and are excluded from location loss.
        """
        sample_info = self.rows[index]
        # Example: a row in L1.json -> dataset/scenes/scene001/
        folder = scene_folder(sample_info, self.data_root)
        request_id = sample_info["request_id"]

        view = sample_info["view"]
        observations = load_observations(folder, views=(view,))
        rgb = observations[view]["rgb"]
        label = load_supervision(folder, request_id, view=view)

        image_tensor = image_to_tensor(rgb)
        request = sample_info["request"]
        color_id = COLORS.index(request["color"])
        pattern_id = PATTERNS.index(request["pattern"])

        image_height, image_width = rgb.shape[:2]
        coordinate_valid = bool(label["coordinate_loss_valid"])
        if coordinate_valid:
            reference_uv = np.asarray(label["reference_uv"])
            image_size = np.array([image_width, image_height])
            normalized_uv = (reference_uv + 0.5) / image_size
        else:
            normalized_uv = np.full(2, np.nan)

        target_uv = torch.tensor(normalized_uv, dtype=torch.float32)
        sample = {
            "image": image_tensor,
            "color": color_id,
            "pattern": pattern_id,
            "uv": target_uv,
            "valid": coordinate_valid,
        }
        return sample
